In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode,expr,col,when,concat,hash

In [29]:
spark = SparkSession.builder.appName("task").getOrCreate()

In [30]:
spark

In [31]:
df_pyspark = spark.read.option('multiline','True').json('/home/milan-thapa/Desktop/Zaki_point_task/files/output/data.json')

In [32]:
df_pyspark.printSchema()

root
 |-- in_network: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- billing_code: string (nullable = true)
 |    |    |-- billing_code_type: string (nullable = true)
 |    |    |-- billing_code_type_version: string (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- negotiated_rates: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |    |    |-- expiration_date: strin

In [33]:
np_data = df_pyspark.selectExpr("*","explode (in_network) as net")
net_file = np_data.withColumn("rates",explode("net.negotiated_rates"))
net_file = net_file.withColumn("prices", explode("rates.negotiated_prices"))
net_file = net_file.withColumn('provider',explode("rates.provider_groups"))
net_file = net_file.withColumn('id',explode("provider.npi"))


network_flat = net_file.selectExpr(
        "net.billing_code",
        "net.billing_code_type",
        "net.negotiation_arrangement",
        "prices.billing_class as billing_class",
        "prices.billing_code_modifier as billing_code_modifier",
        "prices.negotiated_rate as negotiated_rate",
        "prices.negotiated_type as negotiated_type",
        "prices.service_code as service_code",
        'id as npi',
        "provider.tin.type as tin_type",
        "provider.tin.value as tin"

    )
network_flat.printSchema()


root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)



In [34]:
network_flat.show()

+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|       npi|tin_type|       tin|
+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+
|         000|           MS-DRG|                    ffs|institutional|                 NULL|       118400.0|        derived|        [21]|1265410047|     ein|23-1476328|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        16358.0|        derived|        [21]|1437865953|     ein|88-3577015|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        33258.5|        derived|        [21]|1215921457|     ei

In [35]:
provider_cleaned = network_flat.withColumn('tin', expr("REPLACE(tin, '-', '')"))
provider_cleaned.show()

+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+---------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|       npi|tin_type|      tin|
+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+---------+
|         000|           MS-DRG|                    ffs|institutional|                 NULL|       118400.0|        derived|        [21]|1265410047|     ein|231476328|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        16358.0|        derived|        [21]|1437865953|     ein|883577015|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        33258.5|        derived|        [21]|1215921457|     ein|231

In [36]:
provider_replaced = network_flat.withColumn("tin_type",
                    when(col("tin_type") == "ein", 1)
                    .when(col("tin_type") == "npi", 2))
provider_replaced.show()

+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|       npi|tin_type|       tin|
+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+
|         000|           MS-DRG|                    ffs|institutional|                 NULL|       118400.0|        derived|        [21]|1265410047|       1|23-1476328|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        16358.0|        derived|        [21]|1437865953|       1|88-3577015|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        33258.5|        derived|        [21]|1215921457|       

In [37]:
df_combine = network_flat.withColumn("provider_group_id",concat("npi", "tin"))
df_combine.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- service_code: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- npi: long (nullable = true)
 |-- tin_type: string (nullable = true)
 |-- tin: string (nullable = true)
 |-- provider_group_id: string (nullable = true)



In [38]:
df_hash = df_combine.withColumn('provider_group_id',hash("provider_group_id"))
df_hash.show()

+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+-----------------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|billing_code_modifier|negotiated_rate|negotiated_type|service_code|       npi|tin_type|       tin|provider_group_id|
+------------+-----------------+-----------------------+-------------+---------------------+---------------+---------------+------------+----------+--------+----------+-----------------+
|         000|           MS-DRG|                    ffs|institutional|                 NULL|       118400.0|        derived|        [21]|1265410047|     ein|23-1476328|       -425708255|
|         000|           MS-DRG|                    ffs|institutional|                 NULL|        16358.0|        derived|        [21]|1437865953|     ein|88-3577015|        414463782|
|         000|           MS-DRG|                    ffs|instituti